In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.ensemble import IsolationForest

# Load Data
df = pd.read_csv("pi_test.csv")

# 1. Standardize Headers and Types
df.columns = df.columns.str.strip()
df.columns = [c.capitalize() if c.lower() == 'currency' else c for c in df.columns]

# Safe conversion of Date and Amount
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Amount'] = df['Amount'].astype(str).str.replace(',', '').astype(float)

# 2. Currency Normalization to MYR
exchange_rates = {'MYR': 1.0, 'USD': 4.40, 'JPY': 0.033, 'SGD': 3.20}

def normalize_to_myr(row):
    curr = str(row['Currency']).strip().upper()
    rate = exchange_rates.get(curr, 1.0)
    return row['Amount'] * rate

df['Amount'] = df.apply(normalize_to_myr, axis=1)

# 3. Anonymize Company Names
def anonymize(name):
    if not isinstance(name, str): return "Unknown"
    if len(name) < 5: return name
    return name[:3] + "*" * (len(name) - 8) + name[-4:]

df['Company'] = df['Company'].apply(anonymize)

# 4. Cleanup and Removal of Footer Rows
cols_to_drop = ['P.Invoice No', 'Code', 'Ref 1', 'Ext. No', 'Currency', 'Unnamed: 9']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
df = df.dropna(subset=['Amount', 'Date'])

print("Data Pipeline complete. Records processed:", len(df))

In [ ]:
# Identify the last recorded date and set cutoff to the 1st of that month
last_date = df['Date'].max()
cutoff_date = last_date.replace(day=1)

# Filter dataset to include only full months
df_clean = df[df['Date'] < cutoff_date].copy()

print(f"Cliff effect resolved. Cutoff date set to: {cutoff_date}")
print(f"Remaining rows for analysis: {len(df_clean)}")

In [ ]:
df.head(-15)

In [ ]:
df.head(15)

In [ ]:
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month_Name'] = df_clean['Date'].dt.month_name()

# Iterate through years for detailed summary
for year in sorted(df_clean['Year'].unique()):
    yearly_df = df_clean[df_clean['Year'] == year]
    
    total_spend = yearly_df['Amount'].sum()
    top_vendors = yearly_df.groupby('Company')['Amount'].sum().sort_values(ascending=False).head(5)
    
    # Identify busiest month by invoice volume
    busiest_month = yearly_df.groupby('Month_Name')['Amount'].count().idxmax()
    invoice_count = yearly_df.groupby('Month_Name')['Amount'].count().max()
    
    print(f"--- Summary for {year} ---")
    print(f"Total Yearly Spend: RM {total_spend:,.2f}")
    print(f"Busiest Month: {busiest_month} ({invoice_count} Invoices)")
    print("Top Vendors by Spend:")
    print(top_vendors.to_string())
    print("\n")

In [ ]:
# 1. Monthly Spend Trend across all years
plt.figure(figsize=(15, 6))
# Create a continuous monthly timeline
monthly_trend = df_clean.set_index('Date')['Amount'].resample('ME').sum()

plt.plot(monthly_trend.index, monthly_trend.values, color='steelblue', linewidth=2, marker='o')
plt.title('Monthly Spending Trend (All Years)', fontsize=14, fontweight='bold')
plt.ylabel('Total Spend (RM)')
plt.xlabel('Timeline')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Total Spend per Year
yearly_total = df_clean.groupby('Year')['Amount'].sum()

plt.figure(figsize=(10, 6))
bars = yearly_total.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Total Annual Spend', fontsize=14, fontweight='bold')
plt.ylabel('Total Spend (RM)')
plt.xlabel('Year')
plt.xticks(rotation=0)

# Add value labels on top of bars
for bar in bars.patches:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'RM {bar.get_height()/1e6:.1f}M', 
             ha='center', va='bottom', fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
df.head()

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

def advanced_cleaning(text):
    if not isinstance(text, str):
        return ""
    # Convert to uppercase
    text = text.upper()
    # Remove numbers, special characters, and punctuation
    text = re.sub(r'[^A-Z\s]', ' ', text)
    # Remove single characters
    text = re.sub(r'\b[A-Z]\b', ' ', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the enhanced cleaning
df_clean['NLP_Ready_Description'] = df_clean['Description'].apply(advanced_cleaning)

# Expanded Stopword List (Corporate, Administrative, and Junk terms)
stop_words = set([
    'SDN', 'BHD', 'LTD', 'CO', 'INV', 'AND', 'FOR', 'THE', 'WITH', 'FROM',
    'DATE', 'INVOICE', 'PAYMENT', 'REF', 'REMITTANCE', 'NUMBER', 'NO', 'OFF',
    'PTE', 'MALAYSIA', 'MALAYSIAN', 'CORP', 'CORPORATION', 'SERVICES', 'SERVICE',
    'TRADING', 'ENTERPRISE', 'GLOBAL', 'INTERNATIONAL', 'MANAGEMENT', 'SYSTEM',
    'PRODUCTS', 'PRODUCTION', 'DEPARTMENT', 'TOTAL', 'AMOUNT', 'GST', 'TAX'
])

# Tokenize and filter
all_words = ' '.join(df_clean['NLP_Ready_Description']).split()
# Filter out words that are in stop_words or are very short (length <= 2)
filtered_words = [w for w in all_words if w not in stop_words and len(w) > 2]

# Get Top 50 terms
top_50_terms = Counter(filtered_words).most_common(50)

print("Advanced NLP Prep complete. Top 50 frequent terms identified.")
for word, count in top_50_terms[:50]: # Print first 10 for quick verification
    print(f"{word}: {count}")

In [ ]:
# Convert top 50 terms to a DataFrame for plotting
df_top_50 = pd.DataFrame(top_50_terms, columns=['Term', 'Frequency'])

plt.figure(figsize=(12, 12))
sns.barplot(data=df_top_50, x='Frequency', y='Term', palette='Blues_d')
plt.title('Top 50 Frequent Terms in Transaction Descriptions', fontsize=14, fontweight='bold')
plt.xlabel('Frequency (Count)')
plt.ylabel('Keyword')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def clean_for_nlp(text):
    if not isinstance(text, str): return ""
    # Remove special characters and numbers
    text = re.sub(r'[^A-Za-z\s]', '', text).upper()
    return text

df_clean['NLP_Ready_Description'] = df_clean['Description'].apply(clean_for_nlp)

# Simple frequency check to verify cleaning
all_words = ' '.join(df_clean['NLP_Ready_Description']).split()
stop_words = ['SDN', 'BHD', 'LTD', 'CO', 'INV', 'AND', 'FOR', 'THE']
filtered_words = [w for w in all_words if w not in stop_words and len(w) > 2]

print("NLP Prep complete. Top 10 frequent terms:")
print(Counter(filtered_words).most_common(10))

In [ ]:
# 1. Statistical Z-Score (Simple Outliers)
mean_val = df_clean['Amount'].mean()
std_val = df_clean['Amount'].std()
df_clean['Z_Score'] = (df_clean['Amount'] - mean_val) / (std_val + 1e-9)

# 2. Isolation Forest (ML Anomaly Detection)
# We use both Amount and Z_Score to train the forest
iso = IsolationForest(contamination=0.01, random_state=42)
df_clean['ML_Result'] = iso.fit_predict(df_clean[['Amount', 'Z_Score']])
df_clean['ML_Anomaly'] = df_clean['ML_Result'].apply(lambda x: 'Anomaly' if x == -1 else 'Normal')

# 3. Visualize the Differences
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Z-Score Plot
sns.scatterplot(data=df_clean, x='Amount', y='Z_Score', 
                hue=(df_clean['Z_Score'] > 3), palette={True:'red', False:'grey'}, ax=ax1)
ax1.axhline(3, color='black', linestyle='--')
ax1.set_title('Z-Score Detection (Outliers > 3 Std Dev)')

# Isolation Forest Plot
sns.scatterplot(data=df_clean, x='Amount', y='Z_Score', 
                hue='ML_Anomaly', palette={'Anomaly':'orange', 'Normal':'grey'}, ax=ax2)
ax2.set_title('Isolation Forest Detection (ML Patterns)')

plt.tight_layout()
plt.show()

# Print specific differences
only_ml = df_clean[(df_clean['ML_Anomaly'] == 'Anomaly') & (df_clean['Z_Score'] <= 3)]
print(f"Invoices flagged as Anomaly by ML but missed by Z-Score: {len(only_ml)}")

In [ ]:
# 1. Define the logic for Smart Categorization
def apply_smart_cat(row):
    desc = str(row['Description']).upper()
    
    if any(k in desc for k in ['TRAINING', 'COURSE', 'EXCEL', 'SKILLS', 'DEVELOPMENT']):
        return 'HR & Training'
    if any(k in desc for k in ['MEDICAL', 'CLINIC', 'SKHPPA', 'HEALTH', 'HOSPITAL']):
        return 'Staff Welfare'
    if any(k in desc for k in ['GST', 'TAX', 'AUDIT', 'ACCOUNTING', 'SECRETARIAL']):
        return 'Professional Services'
    if any(k in desc for k in ['FORWARDING', 'COURIER', 'TRANSPORT', 'FREIGHT', 'DELIVERY']):
        return 'Logistics'
    if any(k in desc for k in ['MOLD', 'MOULD', 'MACHINE', 'TOOLING']):
        return 'CapEx & Maintenance'
    
    return 'General Operations'

# 2. Apply it to create the main Category column
df_clean['Category_Smart'] = df_clean.apply(apply_smart_cat, axis=1)

# 3. Define the Conflict Detection (Audit for Misclassification)
def check_misclassification(row):
    desc = str(row['Description']).upper()
    cat = row['Category_Smart'] # Using the column we just created
    
    # Conflict Rule 1: Professional fees hidden in Staff Welfare
    if cat == 'Staff Welfare' and any(k in desc for k in ['ADVISORY', 'CONSULTANT', 'FEE']):
        return 'Potential Misclassification: Professional Fee in Welfare'
    
    # Conflict Rule 2: Capital Expenditure (CapEx) hidden in Ops
    if cat == 'General Operations' and any(k in desc for k in ['INSTALLATION', 'CONSTRUCTION', 'ASSET']):
        return 'Potential Misclassification: CapEx in OpEx'
    
    return 'Valid'

# 4. Apply the Audit Flag
df_clean['Audit_Flag'] = df_clean.apply(check_misclassification, axis=1)

print("Categorization and Conflict Audit Complete.")
print("--- Category Distribution ---")
print(df_clean['Category_Smart'].value_counts())
print("\n--- Audit Flag Results ---")
print(df_clean['Audit_Flag'].value_counts())

In [ ]:
df.head()

In [ ]:
df_clean.head()

In [ ]:
# Visualize Misclassifications by Year
plt.figure(figsize=(12, 6))
audit_summary = df_clean[df_clean['Audit_Flag'] != 'Valid'].groupby(['Year', 'Audit_Flag']).size().unstack()
audit_summary.plot(kind='bar', stacked=True, ax=plt.gca(), colormap='Reds')

plt.title('Potential Misclassifications Found per Year', fontsize=14)
plt.ylabel('Count of Suspicious Invoices')
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Risk Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 1. Vendor Consolidation Opportunity (The "Tail Spend" Audit)
In many companies, 20% of vendors account for 80% of spend, while the remaining 80% of vendors (the "tail") create massive administrative work.

The Approach: Group your data by Category and count unique Company names.

The Insight: If you have 15 different vendors for "Training," you are losing bargaining power.

Suggestion: Recommend consolidating these into 1 or 2 preferred vendors to negotiate a 10-15% volume discount.

In [ ]:
# Identify categories with high vendor fragmentation
fragmentation = df_clean.groupby('Category_Smart').agg({
    'Company': 'nunique',
    'Amount': 'sum'
}).rename(columns={'Company': 'Vendor_Count', 'Amount': 'Total_Spend'})

fragmentation['Spend_per_Vendor'] = fragmentation['Total_Spend'] / fragmentation['Vendor_Count']
display(fragmentation.sort_values('Vendor_Count', ascending=False))

# 2. Price Variance Analysis (Unit Cost Forensic)
You have many recurring descriptions (e.g., "MEDICAL SKHPPA" or "EXCEL TRAINING").

The Approach: Filter for identical descriptions and track the Amount over time.

The Insight: Did Vendor A charge RM 136 in 2017 but RM 160 in 2019 for the exact same service?

Suggestion: Highlight "Price Creep" where vendors are slowly raising rates without a formal contract review.

In [ ]:
# Group by Description to find price volatility
# We only look at items that appear at least 5 times to ensure data significance
price_variance = df_clean.groupby('Description').agg({
    'Amount': ['min', 'max', 'mean', 'count', 'std']
})

# Flatten column names
price_variance.columns = ['Min_Price', 'Max_Price', 'Avg_Price', 'Count', 'Std_Dev']

# Calculate the 'Price Creep' percentage
price_variance['Price_Gap_Pct'] = ((price_variance['Max_Price'] - price_variance['Min_Price']) / price_variance['Min_Price']) * 100

# Filter for recurring items (at least 5 times) and significant variance (> 10%)
price_creep_report = price_variance[(price_variance['Count'] >= 5) & (price_variance['Price_Gap_Pct'] > 10)]

print("Top 10 Items with Highest Price Creep:")
display(price_creep_report.sort_values('Price_Gap_Pct', ascending=False).head(10).style.format('{:.2f}'))

In [ ]:
# TEXT OUTPUT / INSIGHT
print("\nSTRATEGIC INSIGHT: PRICE CREEP ANALYSIS")
print("This analysis identifies recurring services where the price has shifted significantly.")
print("A high 'Price_Gap_Pct' suggests that vendors may be increasing rates without a new contract.")
print("Recommendation: Review the top 5 items in this list and verify if these price increases")
print("were officially negotiated or if they are unauthorized 'Price Creep'.")

# 3. "Budget Flushing" & Seasonality detection
Since you fixed the cliff effect, your time-series data is now reliable.

The Approach: Compare spend in Q4 (October–December) against the rest of the year.

The Insight: A massive spike in "Training" or "Maintenance" in December often means departments are spending money just to "use up" their budget so it doesn't get cut next year.

Suggestion: Propose a "Budget Carry-over" policy to reduce wasteful year-end spending.

In [ ]:
# Define Q4 (Oct, Nov, Dec)
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Is_Q4'] = df_clean['Month'].isin([10, 11, 12])

# Compare Category spending Q4 vs Non-Q4
seasonality = df_clean.groupby(['Category_Smart', 'Is_Q4'])['Amount'].sum().unstack()
seasonality.columns = ['Other_Quarters', 'Q4_Spend']

# Calculate the Q4 Intensity (%)
seasonality['Q4_Intensity_Pct'] = (seasonality['Q4_Spend'] / (seasonality['Other_Quarters'] + seasonality['Q4_Spend'])) * 100

# Visualize the spike
plt.figure(figsize=(12, 6))
seasonality['Q4_Intensity_Pct'].sort_values().plot(kind='barh', color='orange')
plt.axvline(25, color='red', linestyle='--', label='Expected 25% Threshold')
plt.title('Budget Flushing Check: % of Total Spend occurring in Q4', fontweight='bold')
plt.xlabel('Percentage of Annual Budget Spent in Q4')
plt.legend()
plt.show()

In [ ]:
# TEXT OUTPUT / INSIGHT
print("STRATEGIC INSIGHT: SEASONALITY & BUDGET FLUSHING")
print("In a perfectly balanced year, Q4 should account for roughly 25% of the total spend.")
print("Categories showing more than 35% spend in Q4 (crossing the red line) are high-risk")
print("for 'Budget Flushing'—spending remaining funds to avoid budget cuts next year.")
print("Recommendation: Implement a 'Carry-over' policy for these specific categories to")
print("discourage wasteful year-end purchasing.")

# 4. Duplicate & Split-Transaction Detection
This is a core forensic audit requirement.

The Approach: Look for transactions where the Company, Amount, and Date (within +/- 2 days) are identical.

The Insight: This catches either duplicate payments (paying the same bill twice) or split invoices (breaking a RM 20,000 purchase into two RM 10,000 invoices to stay under a RM 15,000 approval limit).

Suggestion: Flag these for the internal audit team to verify the physical invoices.

In [ ]:
# --- CORRECTED DUPLICATE AND SPLIT-INVOICE DETECTION ---

# 1. Identify potential split transactions (Same vendor, same day, multiple invoices)
# Fix: Changed 'list' to list (the Python built-in function)
split_logic = df_clean.groupby(['Company', 'Date']).agg({
    'Amount': ['count', 'sum', list] 
})

# Flatten the multi-index columns
split_logic.columns = ['Invoice_Count', 'Total_Daily_Amount', 'Amount_List']

# 2. Filter for cases with more than 1 invoice on the same day from the same vendor
# We sort by 'Total_Daily_Amount' to see the biggest risks first
potential_splits = split_logic[split_logic['Invoice_Count'] > 1].sort_values('Total_Daily_Amount', ascending=False)

print("--- POTENTIAL SPLIT TRANSACTIONS DETECTED ---")
display(potential_splits.head(10))

# 3. Strategic Insight Text Output
print("\nSTRATEGIC INSIGHT: DUPLICATE AND SPLIT-INVOICE DETECTION")
print(f"Total clusters of multiple daily invoices from the same vendor: {len(potential_splits)}")
print("This analysis catches 'Split Invoices' where a large purchase is broken into")
print("smaller amounts to stay under a manager's approval threshold (e.g., RM 10,000).")
print("Recommendation: Pass this list to the Internal Audit team to verify if these")
print("were legitimate separate purchases or an attempt to bypass internal controls.")

# 5. Benford’s Law (Digit Frequency Analysis)
This is a world-class standard for fraud detection in large datasets.

The Approach: Analyze the first digit of every Amount (e.g., in 9000.0, the digit is 9).

The Insight: According to Benford's Law, the digit '1' should appear ~30% of the time. If your data shows a massive spike in '7' or '9', it suggests human manipulation (someone is manually making up numbers).

Suggestion: Use this to identify which specific categories (e.g., "General Operations") have the highest risk of manual data entry errors or fraud.

In [ ]:
# Function to extract the first digit
def get_first_digit(n):
    s = str(abs(n)).replace('.', '').lstrip('0')
    return int(s[0]) if s else None

df_clean['First_Digit'] = df_clean['Amount'].apply(get_first_digit)
actual_dist = df_clean['First_Digit'].value_counts(normalize=True).sort_index()

# Theoretical Benford distribution
benford_dist = [np.log10(1 + 1/d) for d in range(1, 10)]

# Plotting the Audit
plt.figure(figsize=(10, 6))
plt.bar(range(1, 10), actual_dist[:9], alpha=0.7, label='Actual Invoice Data', color='navy')
plt.plot(range(1, 10), benford_dist, marker='o', color='red', linewidth=2, label='Benford Law (Expected)')
plt.xticks(range(1, 10))
plt.title('Benford’s Law Audit: First Digit Frequency', fontsize=14, fontweight='bold')
plt.xlabel('First Digit')
plt.ylabel('Frequency (%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# Strategic Suggestion: Check categories that deviate most
print("Audit Note: If red line and blue bars diverge significantly, data may be manually manipulated.")

In [ ]:
# TEXT OUTPUT / INSIGHT
print("STRATEGIC INSIGHT: DIGITAL INTEGRITY (BENFORD'S LAW)")
print("Benford's Law states that in natural financial data, the digit '1' appears most often.")
print("If the blue bars deviate significantly from the red line (especially at digits 7, 8, or 9),")
print("it indicates that the data may contain man-made or 'forced' numbers.")
print("Result: Significant deviations suggest manual intervention, rounding errors,")
print("or potential fraudulent entries in the accounting system.")

In [ ]:
df_clean.head(-5)

# K-Means Clustering (Vendor Behavior Profiling)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Feature Engineering for Vendors
vendor_behavior = df_clean.groupby('Company').agg({
    'Amount': ['count', 'sum', 'mean']
}).reset_index()
vendor_behavior.columns = ['Company', 'Invoice_Count', 'Total_Spend', 'Avg_Invoice_Value']

# 2. Scaling data for K-Means
scaler = StandardScaler()
features_scaled = scaler.fit_transform(vendor_behavior[['Invoice_Count', 'Total_Spend', 'Avg_Invoice_Value']])

# 3. ML Clustering (Groups vendors into 3 behavioral profiles)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
vendor_behavior['Cluster'] = kmeans.fit_predict(features_scaled)

# 4. Visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(data=vendor_behavior, x='Invoice_Count', y='Total_Spend', 
                hue='Cluster', palette='viridis', size='Avg_Invoice_Value', sizes=(20, 200))
plt.yscale('log') # Log scale helps see high-spend outliers
plt.title('ML Vendor Profiling: Administrative Burden vs. Financial Impact', fontsize=14)
plt.xlabel('Number of Invoices (Process Burden)')
plt.ylabel('Total Annual Spend (Financial Impact)')
plt.show()

# 5. Text Result Output
print("STRATEGIC INSIGHT: VENDOR BEHAVIOR PROFILES")
for i in range(3):
    c_df = vendor_behavior[vendor_behavior['Cluster'] == i]
    print(f"Cluster {i}: {len(c_df)} Vendors | Avg Spend: RM {c_df['Total_Spend'].mean():,.2f}")
print("Action: Clusters with high Invoice Counts but low Total Spend are candidates for 'Procure-to-Pay' optimization.")

# Service Co-occurrence Analysis (Market Basket)

In [ ]:
# 1. Create a matrix of Category Co-occurrence
# We identify instances where multiple categories are billed by one vendor on the same date
basket = df_clean.groupby(['Company', 'Date'])['Category_Smart'].unique()
basket_multi = [list(items) for items in basket if len(items) > 1]

# 2. Build Co-occurrence Heatmap
categories = sorted(df_clean['Category_Smart'].unique())
matrix = pd.DataFrame(0, index=categories, columns=categories)

for items in basket_multi:
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            matrix.loc[items[i], items[j]] += 1
            matrix.loc[items[j], items[i]] += 1

# 3. Visualization
plt.figure(figsize=(10, 8))
sns.heatmap(matrix, annot=True, cmap='YlGnBu', fmt='g')
plt.title('Market Basket: Category Co-occurrence (Bundled Services)', fontsize=14)
plt.show()

# 4. Text Result Output
print("STRATEGIC INSIGHT: BUNDLED SERVICE DETECTION")
print("This heatmap shows which services are frequently invoiced together.")
print("Strong links (high numbers) suggest categories that could be negotiated as a single 'Service Level Agreement' (SLA) rather than separate line items.")

# Time-Series Forecasting (ARIMA)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1. Prepare Monthly Time Series
monthly_data = df_clean.set_index('Date')['Amount'].resample('ME').sum()

# 2. Apply ARIMA Model (Auto-Regressive Integrated Moving Average)
# Order (1,1,1) is a standard starting point for business trends
model = SARIMAX(monthly_data, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
model_results = model.fit(disp=False)

# 3. Forecast 6 Months ahead
forecast = model_results.get_forecast(steps=6)
forecast_df = forecast.summary_frame()

# 4. Visualization
plt.figure(figsize=(12, 6))
plt.plot(monthly_data.index, monthly_data, label='Historical Spend', color='navy', marker='o')
plt.plot(forecast_df.index, forecast_df['mean'], label='Forecast', color='red', linestyle='--', marker='s')
plt.fill_between(forecast_df.index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], color='red', alpha=0.1, label='Confidence Interval')
plt.title('Predictive Spend Forecast: Next 6 Months', fontsize=14)
plt.ylabel('Total Spend (RM)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 5. Text Result Output
next_month = forecast_df.iloc[0]
print(f"FORECAST INSIGHT: Next Month Predicted Spend is RM {next_month['mean']:,.2f}.")
print(f"Range of Expected Spend: RM {next_month['mean_ci_lower']:,.2f} to RM {next_month['mean_ci_upper']:,.2f}.")
print("Action: If actual spend exceeds the 'Confidence Interval', it should trigger an automatic Management Review.")

In [ ]:
# RUN THIS TO SEE IF WE HAVE VARIANCE POTENTIAL
summary = df_clean.groupby('Category_Smart').agg({
    'Amount': ['sum', 'mean', 'max', 'min']
})
summary.columns = ['Total_Spend', 'Avg_Invoice', 'Max_Single', 'Min_Single']
summary['Potential_Saving_Opportunity'] = summary['Total_Spend'] * 0.05 # Assuming a 5% negotiation target
display(summary)

In [ ]:
df_clean.head(30)

In [ ]:
# Identify top 10 vendors by total spend and their volatility
vendor_audit = df_clean.groupby('Company').agg({
    'Amount': ['sum', 'count', 'std', 'mean']
})
vendor_audit.columns = ['Total_Spend', 'Invoice_Count', 'Price_Volatility', 'Avg_Bill']

# Calculate a "Risk Score" (High Volatility + High Spend = High Risk)
vendor_audit['Risk_Score'] = (vendor_audit['Price_Volatility'] / vendor_audit['Avg_Bill']) * vendor_audit['Total_Spend']

print("--- TOP 10 HIGH-RISK VENDORS FOR BUDGET OVERRUNS ---")
display(vendor_audit.sort_values('Risk_Score', ascending=False).head(10))

# STRATEGIC INSIGHT
print("\nSTRATEGIC INSIGHT: VENDOR RISK TRACKING")
print("Vendors with high 'Risk_Score' are those where prices fluctuate wildly despite high spend.")
print("Recommendation: Move these vendors from 'Open Purchase Orders' to 'Fixed-Price Contracts'.")

In [ ]:
# 1. Sort by Company and Date
df_sorted = df_clean.sort_values(['Company', 'Date'])

# 2. Calculate time since last invoice for each vendor
df_sorted['Prev_Date'] = df_sorted.groupby('Company')['Date'].shift(1)
df_sorted['Days_Since_Last'] = (df_sorted['Date'] - df_sorted['Prev_Date']).dt.days

# 3. Flag 'Awakened' vendors (Inactive > 180 days, then a payment > RM 1000)
dormant_risk = df_sorted[(df_sorted['Days_Since_Last'] > 180) & (df_sorted['Amount'] > 1000)]

plt.figure(figsize=(10, 5))
sns.histplot(dormant_risk['Days_Since_Last'], bins=20, color='purple')
plt.title('Dormant Vendor Awakening (Gap in Days before New Payment)', fontweight='bold')
plt.show()

print("--- DORMANT VENDOR ALERTS ---")
display(dormant_risk[['Date', 'Company', 'Description', 'Category_Smart', 'Amount', 'Days_Since_Last']].head(10))

# Result Output
print("\nPRO-MODE INSIGHT: ACCOUNT INTEGRITY")
print("These vendors were inactive for over 6 months before receiving a significant payment.")
print("Action: Verify if these vendors are still active and if their banking details changed recently.")

In [ ]:
# 1. Identify 'Vague' descriptions
vague_keywords = ['GENERAL', 'MISC', 'OTHER', 'FEES', 'CHARGES', 'SERVICE', 'CONSULTANCY']
df_clean['Is_Vague'] = df_clean['Description'].str.upper().apply(lambda x: any(k in str(x) for k in vague_keywords))

# 2. Audit Vague descriptions for price volatility
vague_audit = df_clean[df_clean['Is_Vague']].groupby(['Company', 'Description']).agg({
    'Amount': ['count', 'mean', 'std']
}).reset_index()
vague_audit.columns = ['Company', 'Description', 'Count', 'Avg_Price', 'Price_Volatility']

# 3. Filter for high-risk vague billing
high_risk_vague = vague_audit[(vague_audit['Count'] > 3) & (vague_audit['Price_Volatility'] > (vague_audit['Avg_Price'] * 0.3))]

print("--- VAGUE BILLING FORENSICS (High Volatility in Generic Items) ---")
display(high_risk_vague.sort_values('Price_Volatility', ascending=False).head(10))

# Result Output
print("\nPRO-MODE INSIGHT: ACCOUNTABILITY GAP")
print("These items use generic descriptions but the price fluctuates by more than 30%.")
print("When descriptions are vague, it is easy to overcharge without being noticed.")
print("Action: Enforce a 'No Detail, No Pay' policy for these specific vendors.")

In [ ]:
# 1. Calculate the 'Heartbeat' (average days between invoices) for each vendor
df_sorted = df_clean.sort_values(['Company', 'Date'])
df_sorted['Days_Since_Last'] = df_sorted.groupby('Company')['Date'].diff().dt.days

vendor_heartbeat = df_sorted.groupby('Company').agg({
    'Days_Since_Last': ['mean', 'std', 'count'],
    'Amount': 'sum'
}).reset_index()
vendor_heartbeat.columns = ['Company', 'Avg_Gap', 'Gap_Std', 'Invoice_Count', 'Total_Spend']

# 2. Assign Persona
def assign_persona(row):
    if row['Invoice_Count'] <= 2: return 'Ad-hoc / New'
    if row['Avg_Gap'] <= 45: return 'Continuous (Monthly/Weekly)'
    if 80 <= row['Avg_Gap'] <= 100: return 'Cyclical (Quarterly)'
    if 300 <= row['Avg_Gap'] <= 380: return 'Cyclical (Annual - e.g. Training)'
    return 'Irregular Pattern'

vendor_heartbeat['Vendor_Persona'] = vendor_heartbeat.apply(assign_persona, axis=1)

print("--- VENDOR PERSONA DISTRIBUTION ---")
print(vendor_heartbeat['Vendor_Persona'].value_counts())

# Insight: Now we know WHO should be billing frequently and who shouldn't.

In [ ]:
# 3. Merge persona back to main data
df_pro = df_sorted.merge(vendor_heartbeat[['Company', 'Vendor_Persona', 'Avg_Gap']], on='Company')

# 4. Pro-Mode Logic: Flag 'True' Dormancy
# Flag if the gap is 3x their normal heartbeat AND the persona is 'Continuous'
df_pro['Pro_Dormant_Flag'] = (
    (df_pro['Vendor_Persona'] == 'Continuous (Monthly/Weekly)') & 
    (df_pro['Days_Since_Last'] > (df_pro['Avg_Gap'] * 3)) & 
    (df_pro['Amount'] > 1000)
)

print("--- INTELLIGENT DORMANCY ALERTS (Continuous Vendors only) ---")
display(df_pro[df_pro['Pro_Dormant_Flag'] == True][['Date', 'Company', 'Vendor_Persona', 'Amount', 'Days_Since_Last']].head(10))

# Result: This ignores Training companies (Annual) and only catches broken patterns in operational vendors.

In [ ]:
# Core Logic: Calculating the Coefficient of Variation (CV) for each vendor.
# CV = Standard Deviation / Mean. 
# A higher CV means the pricing is chaotic, inconsistent, and potentially inflated.

# Aggregating data by Company and their behavioral Persona
vendor_cv = df_pro.groupby(['Company', 'Vendor_Persona']).agg({
    'Amount': ['std', 'mean', 'count', 'sum']
}).reset_index()

# Renaming columns for clarity
vendor_cv.columns = ['Company', 'Persona', 'Std_Dev', 'Mean_Price', 'Count', 'Total_Spend']
vendor_cv['CV'] = vendor_cv['Std_Dev'] / vendor_cv['Mean_Price']

# Filtering: Selecting "Irregular Pattern" vendors with more than 3 transactions 
# and extreme price volatility (CV > 0.5)
fraud_risk_list = vendor_cv[
    (vendor_cv['Persona'] == 'Irregular Pattern') & 
    (vendor_cv['Count'] > 3) & 
    (vendor_cv['CV'] > 0.5)
].sort_values('Total_Spend', ascending=False)

# Displaying Top 10 Forensic Alerts
display(fraud_risk_list.head(10))

# MANAGEMENT SUMMARY:
# Finding: Detected multiple irregular vendors with chaotic pricing logic.
# Interpretation: This volatility typically implies:
# 1. Lack of Master Service Agreements (MSAs).
# 2. "Price Discrimination" (Vendor charging based on perceived urgency).
# 3. Potential over-billing/padding of invoices.

In [ ]:
# 1. Isolate the target vendor
target_vendor = "OPT****************** BHD" 
vendor_data = df_clean[df_clean['Company'] == target_vendor].sort_values('Date')

# 2. Plotting the visualization
plt.figure(figsize=(14, 7))
plt.plot(vendor_data['Date'], vendor_data['Amount'], marker='o', linestyle='-', color='red', label='Invoice Amount')

# 3. Annotate the descriptions to see the "Why" behind the spikes
for i, row in vendor_data.iterrows():
    plt.annotate(row['Description'][:20], # Showing first 20 chars of description
                 (row['Date'], row['Amount']),
                 textcoords="offset points", 
                 xytext=(0,10), 
                 ha='center', 
                 fontsize=8, 
                 rotation=45)

plt.title(f'Forensic Price Timeline: {target_vendor}', fontsize=16, fontweight='bold', color='darkred')
plt.xlabel('Transaction Date')
plt.ylabel('Amount (RM)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 4. Truthful Insight for your Boss:
avg = vendor_data['Amount'].mean()
max_val = vendor_data['Amount'].max()
min_val = vendor_data['Amount'].min()

print(f"--- CEO SUMMARY FOR {target_vendor} ---")
print(f"Price Range: RM {min_val:,.2f} to RM {max_val:,.2f}")
print(f"Variance: The highest price is {max_val/min_val:.1f}x higher than the lowest.")
print("Recommendation: Review the highlighted 'peaks' to see if the service scope actually changed.")

In [ ]:
# 1. Isolate the vendor
target_vendor = "OPT****************** BHD" # Use the full name from your list
vendor_detail = df_clean[df_clean['Company'] == target_vendor].copy()

# 2. Extract the extremes
peaks = vendor_detail.nlargest(5, 'Amount')
valleys = vendor_detail.nsmallest(5, 'Amount')

print(f"--- [TOP 5 PEAKS] Why are these so expensive? ---")
display(peaks[['Date', 'Amount', 'Description', 'Category_Smart']])

print(f"\n--- [BOTTOM 5 VALLEYS] What is the 'Base' service? ---")
display(valleys[['Date', 'Amount', 'Description', 'Category_Smart']])

In [ ]:
# Goal: Identify categories with many unique vendors but small average transaction values.
category_efficiency = df_clean.groupby('Category_Smart').agg({
    'Company': 'nunique',
    'Amount': ['count', 'sum', 'mean']
}).reset_index()

category_efficiency.columns = ['Category', 'Unique_Vendors', 'Total_Invoices', 'Total_Spend', 'Avg_Invoice_Value']

# Calculate the "Fragmentation Index":
# A higher index means the category is "shattered" into too many small suppliers.
category_efficiency['Fragmentation_Index'] = (category_efficiency['Unique_Vendors'] / category_efficiency['Total_Spend']) * 1000000

print("--- Procurement Fragmentation Analysis: Candidates for Vendor Consolidation ---")
display(category_efficiency.sort_values('Fragmentation_Index', ascending=False))

In [ ]:
# 1. Finding Exact Duplicates: Same Vendor, Same Amount, Same Date.
duplicates = df_clean[df_clean.duplicated(subset=['Company', 'Amount', 'Date'], keep=False)]

# 2. Finding "Near-Duplicates": Same Vendor, Same Amount, but Date differs by ≤ 3 days.
# This catches cases where an invoice was re-entered a day later by mistake.
df_dup = df_clean.sort_values(['Company', 'Amount', 'Date'])
df_dup['Diff'] = df_dup.groupby(['Company', 'Amount'])['Date'].diff().dt.days
near_duplicates = df_dup[df_dup['Diff'] <= 3]

print(f"--- Forensic Audit: Detected {len(duplicates) + len(near_duplicates)} Suspected Duplicate Payments ---")

# Combine results for a final check-list
final_report = pd.concat([duplicates, near_duplicates]).drop_duplicates()
display(final_report.head(20))

In [ ]:
# ---------------------------------------------------------
# OFFICE-ONLY FORENSIC SCRIPT: DUPLICATE & RE-ENTRY FINDER
# ---------------------------------------------------------

# 1. Exact Duplicates (Same Vendor, Same Amount, Same Date)
exact_dupes = df_clean[df_clean.duplicated(subset=['Company', 'Amount', 'Date'], keep=False)].copy()

# 2. Potential "Typo" Duplicates (Same Vendor, Same Amount, but Date is slightly off)
# Sort to find sequential entries
df_check = df_clean.sort_values(['Company', 'Amount', 'Date'])
df_check['Days_Diff'] = df_check.groupby(['Company', 'Amount'])['Date'].diff().dt.days

# Flag if the same amount was paid to the same vendor within 31 days
# (Often happens when an invoice is submitted twice in two different months)
near_dupes = df_check[(df_check['Days_Diff'] > 0) & (df_check['Days_Diff'] <= 31)]

# 3. Output for your relative (The "Audit Report")
print(f"REPORT: Found {len(exact_dupes)} exact duplicates.")
print(f"REPORT: Found {len(near_dupes)} potential double-billing cases (within 31 days).")

# Save this to an Excel file ONLY ON YOUR OFFICE PC
# result_report = pd.concat([exact_dupes, near_dupes])
# result_report.to_excel("POTENTIAL_REFUNDS_TO_CLAIM.xlsx")

In [ ]:
# ---------------------------------------------------------
# FINAL AUDIT: CALCULATING RECOVERABLE CASH (FIXED VERSION)
# ---------------------------------------------------------

# 1. Exact Duplicates 
# We identify all "extra" copies of the same transaction
extra_copies = df_clean[df_clean.duplicated(subset=['Company', 'Amount', 'Date'], keep='first')]
exact_refund_value = extra_copies['Amount'].sum()

# 2. Near Duplicates (Double-billing within 31 days)
# We re-run the logic to ensure we only sum the numeric 'Amount'
df_check = df_clean.sort_values(['Company', 'Amount', 'Date'])
df_check['Diff'] = df_check.groupby(['Company', 'Amount'])['Date'].diff().dt.days
near_duplicates_final = df_check[(df_check['Diff'] > 0) & (df_check['Diff'] <= 31)]
near_refund_value = near_duplicates_final['Amount'].sum()

total_at_risk = exact_refund_value + near_refund_value

print(f"--- FINAL FORENSIC SUMMARY (REPAIRED) ---")
print(f"1. Confirmed Exact Duplicates: RM {exact_refund_value:,.2f}")
print(f"2. Potential Double-Billings:  RM {near_refund_value:,.2f}")
print(f"----------------------------------------------")
print(f"TOTAL POTENTIAL RECOVERY:      RM {total_at_risk:,.2f}")

In [ ]:
df_clean.head()

In [ ]:
# ---------------------------------------------------------
# FINAL HANDOVER GENERATOR (UNIVERSAL CSV VERSION)
# ---------------------------------------------------------

# 1. FILE 1: TOP 20 DUPLICATE CLAIMS (The "Cash Recovery" List)
exact_dupes = df_clean[df_clean.duplicated(subset=['Company', 'Amount', 'Date'], keep=False)]
df_sort = df_clean.sort_values(['Company', 'Amount', 'Date'])
df_sort['Diff'] = df_sort.groupby(['Company', 'Amount'])['Date'].diff().dt.days
near_dupes = df_sort[df_sort['Diff'] <= 3]

top_20_refunds = pd.concat([exact_dupes, near_dupes]).drop_duplicates()
top_20_refunds = top_20_refunds.sort_values(by='Amount', ascending=False).head(20)
# Exporting as CSV (Excel can open this!)
top_20_refunds.to_csv("1_Top_20_Duplicate_Claims.csv", index=False)

# 2. FILE 2: VENDOR PERSONA SUMMARY (The "Consolidation" Strategy)
vendor_distribution = vendor_heartbeat['Vendor_Persona'].value_counts().reset_index()
vendor_distribution.columns = ['Persona', 'Vendor_Count']
vendor_distribution.to_csv("2_Vendor_Persona_Summary.csv", index=False)

# 3. FILE 3: HIGH RISK VOLATILITY LIST (The "Negotiation" List)
# Ensure columns exist before exporting
risk_cols = ['Company', 'Persona', 'CV', 'Total_Spend', 'Count']
available_risk_cols = [c for c in risk_cols if c in fraud_risk_list.columns]
high_risk_list = fraud_risk_list.sort_values(by='CV', ascending=False).head(20)
high_risk_list[available_risk_cols].to_csv("3_High_Volatility_Negotiation_List.csv", index=False)

print("--- SUCCESS ---")

In [ ]:
# View the Top 20 Duplicates (The "Money" List)
print("--- PREVIEW: TOP 20 DUPLICATES ---")
df_top_20 = pd.read_csv("1_Top_20_Duplicate_Claims.csv")
display(df_top_20)

# View the Vendor Strategy Summary
print("\n--- PREVIEW: VENDOR PERSONAS ---")
df_personas = pd.read_csv("2_Vendor_Persona_Summary.csv")
display(df_personas)

# View the Negotiation/Volatility List
print("\n--- PREVIEW: HIGH VOLATILITY VENDORS ---")
df_volatility = pd.read_csv("3_High_Volatility_Negotiation_List.csv")
display(df_volatility)

In [ ]:
import matplotlib.pyplot as plt

# Calculate Cumulative Spend
spend_plot = vendor_heartbeat.sort_values('Total_Spend', ascending=False)
spend_plot['Cum_Percentage'] = 100 * spend_plot['Total_Spend'].cumsum() / spend_plot['Total_Spend'].sum()

plt.figure(figsize=(10, 6))
plt.plot(range(len(spend_plot)), spend_plot['Cum_Percentage'], color='blue', linewidth=2)
plt.axhline(y=80, color='red', linestyle='--', label='80% of Spend')
plt.title('Spend Concentration: Where is the money going?', fontsize=14)
plt.xlabel('Number of Vendors')
plt.ylabel('Cumulative % of Total Spend')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Pro-Insight:
top_10_count = int(len(spend_plot) * 0.1)
top_10_spend = spend_plot.iloc[:top_10_count]['Total_Spend'].sum() / spend_plot['Total_Spend'].sum() * 100
print(f"Insight: Your top 10% of vendors account for {top_10_spend:.1f}% of your total spend.")

In [ ]:
# 1. Sort vendors by spend
vip_list = vendor_heartbeat.sort_values('Total_Spend', ascending=False)

# 2. Calculate how many vendors make up the top 10%
top_10_count = int(len(vip_list) * 0.10)

# 3. Slice the top 10% and select key columns for the boss
top_vendors_report = vip_list.head(top_10_count)[['Company', 'Total_Spend', 'Vendor_Persona', 'Invoice_Count']]

# 4. Add a "Percentage of Total" column for extra impact
total_company_spend = vendor_heartbeat['Total_Spend'].sum()
top_vendors_report['Percentage_of_Total'] = (top_vendors_report['Total_Spend'] / total_company_spend) * 100

print(f"--- THE POWER PLAYERS: Top {top_10_count} Vendors (92.1% of Spend) ---")
display(top_vendors_report)

# 5. Export this to your handover folder
top_vendors_report.to_csv("4_Top_10_Percent_VIP_Vendors.csv", index=False)

In [ ]:
df.head()

In [ ]:
df_clean.head()